<a href="https://colab.research.google.com/github/MuhammadAli055/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

# Clone your repo
!git clone https://github.com/MuhammadAli055/flyrank-ml-internship.git
os.chdir('/content/flyrank-ml-internship')

# Install required packages
!pip install duckdb huggingface_hub scikit-learn -q

print("Setup complete!")
print(os.listdir())

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 108 (delta 26), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 1.85 MiB | 5.97 MiB/s, done.
Resolving deltas: 100% (26/26), done.
Setup complete!
['LICENSE', 'CLAUDE.md', 'submission', 'data', 'work', 'outputs', 'requirements.txt', '.git', 'AGENTS.md', 'README.md', 'notebooks', 'skills', 'SETUP.md', 'DATA_USE.md', '.gitignore', '.github', 'GUIDE.md', 'docs', 'scripts']


In [3]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from huggingface_hub import login

# Get HF token from Colab Secrets (never paste tokens directly!)
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("HuggingFace login successful!")

# Set up DuckDB with HuggingFace access
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Create HuggingFace secret for DuckDB
con.execute(f"""
    CREATE SECRET IF NOT EXISTS hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

# Define paths — using mid-panel month (March 2026), NOT the final month
FACT_MONTH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

print("DuckDB connection ready!")

HuggingFace login successful!
DuckDB connection ready!


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Section 1: My Data Contract — Five Plain Answers

### 1. What does one row mean in my lane?
One row = one content page, aggregated over a defined time window (one month).
At the raw fact table level: one row = one report_date × one client × one content item.
After aggregation for Lane 2: one row = one content page × one month.

### 2. Which table(s) will I use?
Primary table: fact_content_daily_performance
- The daily fact table: 78.8 million rows, report_date from 2025-01-27 to 2026-06-30
- I aggregate it by content_hash_id and client_hash_id over a chosen month window

Supporting table: dim_content (for content metadata like content type and age in later weeks)

Working month for all development: month = 2026-03 (mid-panel month, safely away from the final month)

### 3. Which time window?
Development window: March 2026 (month = 2026-03)
I deliberately avoid June 2026 — the final month — because it is the natural outcome
window for any future-looking label. Using it to develop would mean the test month
and the development month are the same, which defeats honest validation.

### 4. What would I predict or rank (label or proxy)?
Proxy label: is_declining_label
A page is labeled 1 (declining) if its impressions in the second half of March dropped
more than 20% compared to the first half of March. Otherwise it is labeled 0.

Formula: is_declining_label = 1 if (impressions_second_half / impressions_first_half) < 0.8 else 0

This is a proxy — a stand-in for a proper future-window label. A stronger version would
use features from one window (e.g. January–February) to predict outcomes in a later window
(e.g. April–May), with no overlap. I will build that in Week 5.

### 5. One thing I deliberately exclude
I exclude all pages where total impressions for the month = 0.
Pages with zero impressions have no search visibility signal at all.
Including them would flood the dataset with pages that are simply invisible —
not declining — and would make the declining label meaningless.
The filter HAVING SUM(impressions) > 0 in my query enforces this exclusion.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [11]:
feature_query = f"""
WITH monthly_agg AS (
    SELECT
        content_hash_id,
        client_hash_id,

        -- Feature 1: Total impressions for the month
        SUM(gsc_impressions) as impressions_monthly,

        -- Feature 2: Average search position
        ROUND(AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END), 2) as avg_position,

        -- Feature 3: Click-Through Rate
        ROUND(
            CASE WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE 0 END,
        4) as ctr,

        -- Feature 4: Total GA4 sessions
        SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_organic ELSE 0 END)
            as sessions_monthly,

        -- Feature 5: Days active
        COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END) as days_with_impressions,

        -- For proxy label: first half vs second half of March
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END)
            as impressions_first_half,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END)
            as impressions_second_half

    FROM read_parquet('{FACT_MONTH}')
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
)
SELECT
    content_hash_id,
    client_hash_id,
    impressions_monthly,
    avg_position,
    ctr,
    sessions_monthly,
    days_with_impressions,

    -- Proxy label
    CASE
        WHEN impressions_first_half > 0
        AND (impressions_second_half * 1.0 / impressions_first_half) < 0.8
        THEN 1
        ELSE 0
    END as is_declining_label,

    impressions_first_half,
    impressions_second_half

FROM monthly_agg
ORDER BY impressions_monthly DESC
"""

feature_df = con.execute(feature_query).df()

print(f"Feature frame shape: {feature_df.shape}")
print(f"One row = one content page, aggregated over March 2026\n")
print(f"Declining pages: {feature_df['is_declining_label'].sum()} "
      f"({feature_df['is_declining_label'].mean()*100:.1f}%)")
print(f"\nFirst 5 rows:")
print(feature_df[['content_hash_id','impressions_monthly','avg_position',
                   'ctr','sessions_monthly','days_with_impressions',
                   'is_declining_label']].head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 10)
One row = one content page, aggregated over March 2026

Declining pages: 49673 (28.1%)

First 5 rows:
            content_hash_id  impressions_monthly  avg_position     ctr  \
0  content_eadb33b5df496f4a             617124.0          2.38  0.0092   
1  content_ec2e0346994fb5a5             245276.0          2.85  0.0060   
2  content_e8a52cf3d5988c07             244931.0         15.01  0.0027   
3  content_0e03de7680314cd5             221310.0          2.68  0.0033   
4  content_44f34c0a90047651             212404.0          7.35  0.0001   

   sessions_monthly  days_with_impressions  is_declining_label  
0            3171.0                     29                   0  
1             979.0                     29                   0  
2             942.0                     31                   1  
3             648.0                     29                   0  
4              33.0                     31                   0  


## Five Features — "Knowable at the Decision Moment Because..."

| Feature | What it measures | Knowable at decision moment because... |
|---|---|---|
| impressions_monthly | Total times the page appeared in search results in March | Past search impressions, fully recorded in Google Search Console before any decision |
| avg_position | Average Google ranking position across all days in March | Position is a past observed fact from GSC, known before we act on any page |
| ctr | Clicks divided by impressions for the month | Both clicks and impressions are past observations, fully recorded before the decision point |
| sessions_monthly | Total GA4 website visits in March (GA4-tracked rows only) | GA4 sessions are past events already recorded; IS TRUE filter ensures we only use rows with confirmed tracking |
| days_with_impressions | How many days in March the page had at least 1 impression | Derived from past daily records, fully known before the decision moment |

None of these features require any knowledge of the future. All five are aggregated
from March 2026 observations and are available to a reviewer making a decision at
the end of March — before anything is done to any page.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# ============================================================
# QUERY 1: Grain Check
# Prove that one row = one report_date + client_hash_id + content_hash_id
# ============================================================
query1 = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT
        CONCAT(CAST(report_date AS VARCHAR), '|', client_hash_id, '|', content_hash_id)
    ) as unique_day_client_page_combinations
FROM read_parquet('{FACT_MONTH}')
"""

result1 = con.execute(query1).df()
print("=== QUERY 1: GRAIN CHECK ===")
print(result1)
print("\nIf total_rows == unique_day_client_page_combinations, grain is confirmed ✅")
print("One row = one report_date × one client × one content page")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== QUERY 1: GRAIN CHECK ===
   total_rows  unique_day_client_page_combinations
0     9841378                              9841378

If total_rows == unique_day_client_page_combinations, grain is confirmed ✅
One row = one report_date × one client × one content page


In [6]:
# ============================================================
# QUERY 2: Row Count and Date Span of My Slice
# ============================================================
query2 = f"""
SELECT
    COUNT(*)                        as total_rows,
    MIN(report_date)                as earliest_date,
    MAX(report_date)                as latest_date,
    COUNT(DISTINCT client_hash_id)  as unique_clients,
    COUNT(DISTINCT content_hash_id) as unique_content_pages
FROM read_parquet('{FACT_MONTH}')
"""

result2 = con.execute(query2).df()
print("=== QUERY 2: ROW COUNT AND DATE SPAN ===")
print(result2)

=== QUERY 2: ROW COUNT AND DATE SPAN ===
   total_rows earliest_date latest_date  unique_clients  unique_content_pages
0     9841378    2026-03-01  2026-03-31              55                331437


In [8]:
# ============================================================
# QUERY 3: Availability Check using IS TRUE
# ============================================================
query3 = f"""
SELECT
    COUNT(*)                                                        as total_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END)         as rows_with_ga4_data,
    COUNT(CASE WHEN gsc_impressions IS NOT NULL
               AND gsc_impressions > 0 THEN 1 END)                 as rows_with_impressions,
    ROUND(
        100.0 * COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END)
        / COUNT(*), 2
    )                                                               as pct_with_ga4
FROM read_parquet('{FACT_MONTH}')
"""

result3 = con.execute(query3).df()
print("=== QUERY 3: AVAILABILITY CHECK (IS TRUE) ===")
print(result3)
print("\nRows that survive ga4_data_available IS TRUE filter are the ones with")
print("both search AND analytics data — the strongest signal for Lane 2.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== QUERY 3: AVAILABILITY CHECK (IS TRUE) ===
   total_rows  rows_with_ga4_data  rows_with_impressions  pct_with_ga4
0     9841378              413966                3611061          4.21

Rows that survive ga4_data_available IS TRUE filter are the ones with
both search AND analytics data — the strongest signal for Lane 2.


## What The Three Queries Proved

Query 1 — Grain confirmed: total rows matched unique day×client×page combinations,
proving the table truly has one row per day per client per content item.

Query 2 — Date span confirmed: my March 2026 slice covers the full month,
with a known number of unique clients and content pages.

Query 3 — Availability confirmed using IS TRUE: only a subset of rows have GA4
data available. This is the filter I must apply before using any session-based
features, otherwise I would treat "no tracking yet" as "no traffic", which would
be a silent, dangerous mistake.

In [12]:
# ============================================================
# THE LEAKAGE TRAP — Step 1: Honest features, honest score
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Work with clean data only
df_model = feature_df.dropna().copy()

y = df_model['is_declining_label']

# HONEST FEATURES — no leakage
honest_features = [
    'impressions_monthly',
    'avg_position',
    'ctr',
    'sessions_monthly',
    'days_with_impressions'
]

X_honest = df_model[honest_features]

scaler = StandardScaler()
X_honest_scaled = scaler.fit_transform(X_honest)

lr = LogisticRegression(random_state=42)
honest_auc = cross_val_score(lr, X_honest_scaled, y, cv=3, scoring='roc_auc').mean()

print("=== STEP 1: HONEST FEATURES (no leakage) ===")
print(f"Features: {honest_features}")
print(f"ROC AUC:  {honest_auc:.3f}")
print("\nThis is our real, honest starting score. Remember this number.")

=== STEP 1: HONEST FEATURES (no leakage) ===
Features: ['impressions_monthly', 'avg_position', 'ctr', 'sessions_monthly', 'days_with_impressions']
ROC AUC:  0.634

This is our real, honest starting score. Remember this number.


In [13]:
# ============================================================
# THE LEAKAGE TRAP — Step 2: Add leaked column (BAD!)
# ============================================================

# THE TRAP:
# Our label is: is_declining_label = 1 if impressions_second_half < 0.8 * impressions_first_half
# So impressions_second_half DIRECTLY determines the label.
# Adding it as a feature means the model reads the answer from the future.
# This is pure leakage.

leaked_features = honest_features + ['impressions_second_half']  # ← THIS IS WRONG

X_leaked = df_model[leaked_features]
X_leaked_scaled = scaler.fit_transform(X_leaked)

leaked_auc = cross_val_score(lr, X_leaked_scaled, y, cv=3, scoring='roc_auc').mean()

print("=== STEP 2: LEAKED FEATURES (WRONG — do not use!) ===")
print(f"Features: {leaked_features}")
print(f"ROC AUC with leak:  {leaked_auc:.3f}")
print(f"\n>>> AUC jumped: {honest_auc:.3f} → {leaked_auc:.3f}")
print(f">>> Fake improvement: +{leaked_auc - honest_auc:.3f}")
print(f"\nThe model is not smarter. It is CHEATING.")
print(f"It is reading impressions_second_half, which IS the answer it is supposed to predict.")
print(f"In the real world, this number does not exist at decision time.")

=== STEP 2: LEAKED FEATURES (WRONG — do not use!) ===
Features: ['impressions_monthly', 'avg_position', 'ctr', 'sessions_monthly', 'days_with_impressions', 'impressions_second_half']
ROC AUC with leak:  0.853

>>> AUC jumped: 0.634 → 0.853
>>> Fake improvement: +0.219

The model is not smarter. It is CHEATING.
It is reading impressions_second_half, which IS the answer it is supposed to predict.
In the real world, this number does not exist at decision time.


In [14]:
# ============================================================
# THE LEAKAGE TRAP — Step 3: Delete the leak, keep honest score
# ============================================================

# Remove the leaking columns from our working frame permanently
feature_df_clean = feature_df.drop(
    columns=['impressions_first_half', 'impressions_second_half']
)

print("=== STEP 3: LEAKAGE REMOVED — CLEAN FRAME ===")
print(f"Columns in clean frame: {list(feature_df_clean.columns)}")
print(f"\nFinal feature frame shape: {feature_df_clean.shape}")
print(f"\nHonest ROC AUC we carry forward: {honest_auc:.3f}")
print(f"Leaked ROC AUC we throw away:    {leaked_auc:.3f}")
print(f"\nLesson learned: if a score looks too good, something probably leaked.")
print(f"Always ask: could this feature exist at the moment the decision is made?")
print(f"If the answer is no — remove it immediately.")

# Show the clean frame
print("\nFinal clean feature frame (first 5 rows):")
print(feature_df_clean.head())

=== STEP 3: LEAKAGE REMOVED — CLEAN FRAME ===
Columns in clean frame: ['content_hash_id', 'client_hash_id', 'impressions_monthly', 'avg_position', 'ctr', 'sessions_monthly', 'days_with_impressions', 'is_declining_label']

Final feature frame shape: (176738, 8)

Honest ROC AUC we carry forward: 0.634
Leaked ROC AUC we throw away:    0.853

Lesson learned: if a score looks too good, something probably leaked.
Always ask: could this feature exist at the moment the decision is made?
If the answer is no — remove it immediately.

Final clean feature frame (first 5 rows):
            content_hash_id           client_hash_id  impressions_monthly  \
0  content_eadb33b5df496f4a  client_e547b89c05043229             617124.0   
1  content_ec2e0346994fb5a5  client_e547b89c05043229             245276.0   
2  content_e8a52cf3d5988c07  client_23a62021009f63c4             244931.0   
3  content_0e03de7680314cd5  client_e547b89c05043229             221310.0   
4  content_44f34c0a90047651  client_23a6202

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Section 4: One Named Limitation of My Slice

My proxy label is not a true future-window label.

The is_declining_label I defined compares the first half of March to the second half
of March — both halves are inside the same month I used to build features. This means
features like impressions_monthly and ctr include data from both halves, including
the half that determines the label.

This is a risk. In a production-quality model I would:
- Use features from an earlier window (e.g. January–February)
- Predict outcomes in a clearly separate later window (e.g. April–May)  
- Never let the feature window and the target window overlap at all

For now, this proxy is acceptable for learning the pipeline mechanics. I will replace
it with a proper forward-looking label in Week 5, when I have enough warehouse history
to define clean, non-overlapping windows with full confidence.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Section 5: Self-Check

- [x] Five plain-words contract answers written: grain, tables, time window, label/proxy, one exclusion
- [x] Exactly three verification queries run with outputs visible
- [x] Availability checked with IS TRUE (ga4_data_available IS TRUE)
- [x] Five-feature frame built from real warehouse data
- [x] Every feature has a clear "knowable at decision moment because..." explanation
- [x] Deliberate leakage experiment shown: AUC jumped from honest → leaked score
- [x] Leaked column deleted, honest AUC carried forward
- [x] One named limitation of my slice stated clearly
- [x] No raw URLs, client names, domains, or private queries appear in any output
- [x] Used careful language throughout: "proxy", "observed", "suggests", not "proves"